# MatPropNet: Materials Property Prediction using Classical ML and Crystal Graph Neural Networks

This notebook builds a clean end-to-end materials informatics pipeline using the JARVIS-DFT dataset. The workflow compares composition-descriptor-based classical machine learning with a crystal graph neural network for predicting formation energy per atom.

**Main stages:**
1. Load and clean JARVIS materials data
2. Generate composition descriptors using Matminer
3. Train an XGBoost baseline model
4. Build crystal graphs from atomic structures
5. Train an improved PyTorch Geometric GNN with early stopping
6. Compare models and save reproducible outputs

## 1. Colab Setup
Run this cell once at the start of the notebook. A GPU runtime is recommended for the GNN section.

In [ ]:
# Core scientific stack
!pip install -q numpy pandas matplotlib scikit-learn scipy tqdm joblib

# Materials science libraries
!pip install -q jarvis-tools pymatgen matminer ase

# Classical ML
!pip install -q xgboost catboost lightgbm

# Explainability support
!pip install -q shap

# Deep learning and graph neural networks
!pip install -q torch torchvision torchaudio torch-geometric

# Optional dashboard support for later GitHub extension
!pip install -q streamlit plotly

## 2. Imports and Reproducibility
This cell imports all required libraries and fixes random seeds for more stable results.

In [ ]:
import os
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor
from jarvis.db.figshare import data
from jarvis.core.atoms import Atoms
from pymatgen.core.composition import Composition

from matminer.featurizers.composition import ElementProperty, Stoichiometry, ValenceOrbital, IonProperty
from matminer.featurizers.base import MultipleFeaturizer

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import NNConv, global_mean_pool, global_max_pool

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Load JARVIS-DFT Dataset
The JARVIS-DFT 3D dataset contains computed crystal structures and materials properties such as formation energy, bandgap, and elastic moduli.

In [ ]:
raw_data = data("dft_3d")
df = pd.DataFrame(raw_data)

print("Total materials:", len(df))
print("Dataset shape:", df.shape)
display(df.head())

## 4. Target Selection and Data Cleaning
Elastic modulus columns may contain string values such as `na`, so all target columns are explicitly converted to numeric format. Invalid rows are removed before modeling.

In [ ]:
target_columns = [
    "formation_energy_peratom",
    "optb88vdw_bandgap",
    "bulk_modulus_kv",
    "shear_modulus_gv",
]

useful_columns = [
    "jid", "formula", "atoms", "density", "spg_number",
    "spg_symbol", "crys", "nat",
] + target_columns

df_work = df[useful_columns].copy()

numeric_columns = ["density", "nat"] + target_columns
for col in numeric_columns:
    df_work[col] = pd.to_numeric(df_work[col], errors="coerce")

print("Missing values before cleaning:")
print(df_work.isna().sum())

df_clean = df_work.dropna(subset=numeric_columns + ["atoms", "formula"]).copy()

df_clean = df_clean[
    (df_clean["density"] > 0)
    & (df_clean["nat"] > 0)
    & (df_clean["bulk_modulus_kv"] > 0)
    & (df_clean["shear_modulus_gv"] > 0)
].reset_index(drop=True)

print("Cleaned dataset shape:", df_clean.shape)
display(df_clean.head())

## 5. Exploratory Target Analysis
These plots help identify skewed targets and outliers. Formation energy is used as the first target because it is a standard benchmark property in materials informatics.

In [ ]:
target_stats = df_clean[target_columns].describe().T
display(target_stats)

for target in target_columns:
    plt.figure(figsize=(7, 4))
    plt.hist(df_clean[target], bins=60)
    plt.xlabel(target)
    plt.ylabel("Count")
    plt.title(f"Distribution of {target}")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 6. Composition Feature Engineering with Matminer
Classical ML models require fixed-length numerical descriptors. Matminer converts chemical formulae into composition-level features such as stoichiometry, electronegativity statistics, valence orbital descriptors, and ionic character features.

In [ ]:
df_clean["composition"] = df_clean["formula"].apply(Composition)

featurizer = MultipleFeaturizer([
    Stoichiometry(),
    ElementProperty.from_preset("magpie"),
    ValenceOrbital(),
    IonProperty(),
])

print("Generating composition descriptors...")
df_featurized = featurizer.featurize_dataframe(
    df_clean.copy(),
    col_id="composition",
    ignore_errors=True,
)

feature_labels = featurizer.feature_labels()
print("Total generated descriptor labels:", len(feature_labels))
print("First 10 descriptors:", feature_labels[:10])

## 7. Clean Feature Matrix and Prevent Target Leakage
Only Matminer-generated descriptor columns are used as inputs. Target columns are intentionally excluded to avoid leakage.

In [ ]:
feature_columns = [col for col in feature_labels if col in df_featurized.columns]
feature_df = df_featurized[feature_columns].select_dtypes(include=[np.number]).copy()

missing_threshold = 0.10 * len(feature_df)
feature_df = feature_df.loc[:, feature_df.isna().sum() < missing_threshold]
feature_df = feature_df.fillna(feature_df.median())

selector = VarianceThreshold(threshold=0)
X_array = selector.fit_transform(feature_df)
selected_columns = feature_df.columns[selector.get_support()]
feature_df = pd.DataFrame(X_array, columns=selected_columns)

print("Final composition feature matrix:", feature_df.shape)

## 8. XGBoost Baseline Model
XGBoost is used as a strong classical ML baseline for formation energy prediction from composition descriptors.

In [ ]:
target = "formation_energy_peratom"
X = feature_df.copy()
y = df_clean[target].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED
)

composition_scaler = StandardScaler()
X_train_scaled = composition_scaler.fit_transform(X_train)
X_test_scaled = composition_scaler.transform(X_test)

xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=SEED,
    n_jobs=-1,
)

print("Training XGBoost baseline...")
xgb_model.fit(X_train_scaled, y_train)
xgb_pred = xgb_model.predict(X_test_scaled)

xgb_mae = mean_absolute_error(y_test, xgb_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
xgb_r2 = r2_score(y_test, xgb_pred)

print("===== XGBoost Results =====")
print(f"MAE  : {xgb_mae:.4f} eV/atom")
print(f"RMSE : {xgb_rmse:.4f}")
print(f"R²   : {xgb_r2:.4f}")

## 9. XGBoost Evaluation and Feature Importance
Parity and residual plots check model bias and error spread. Feature importance gives a first-level interpretation of chemically relevant descriptors.

In [ ]:
def regression_report(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }


def plot_parity(y_true, y_pred, title):
    plt.figure(figsize=(7, 7))
    plt.scatter(y_true, y_pred, alpha=0.5)
    min_val = min(np.min(y_true), np.min(y_pred))
    max_val = max(np.max(y_true), np.max(y_pred))
    plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")
    plt.xlabel("True Formation Energy")
    plt.ylabel("Predicted Formation Energy")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_residuals(y_true, y_pred, title):
    residuals = np.asarray(y_true) - np.asarray(y_pred)
    plt.figure(figsize=(8, 5))
    plt.hist(residuals, bins=60)
    plt.xlabel("Residual Error")
    plt.ylabel("Count")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_parity(y_test.values, xgb_pred, "XGBoost Parity Plot")
plot_residuals(y_test.values, xgb_pred, "XGBoost Residual Distribution")

importance_df = pd.DataFrame({
    "Feature": feature_df.columns,
    "Importance": xgb_model.feature_importances_,
}).sort_values("Importance", ascending=False)

display(importance_df.head(20))

plt.figure(figsize=(10, 8))
top_features = importance_df.head(20)
plt.barh(top_features["Feature"][::-1], top_features["Importance"][::-1])
plt.xlabel("Importance")
plt.title("Top XGBoost Features")
plt.tight_layout()
plt.show()

## 10. Crystal Graph Construction
For the GNN, each crystal is represented as a graph: atoms are nodes, neighboring atoms are edges, distances are edge attributes, and formation energy is the graph target.

In [ ]:
def structure_to_graph(atoms_dict, target_value, cutoff=5.0):
    # Convert a JARVIS atoms dictionary into a PyTorch Geometric graph.
    atoms_obj = Atoms.from_dict(atoms_dict)
    structure = atoms_obj.pymatgen_converter()

    atomic_numbers = [site.specie.Z for site in structure]
    z = torch.tensor(atomic_numbers, dtype=torch.long)

    edge_index = []
    edge_attr = []

    for i, site_i in enumerate(structure):
        neighbors = structure.get_neighbors(site_i, cutoff)
        for neighbor in neighbors:
            j = neighbor.index
            dist = neighbor.nn_distance
            if i != j:
                edge_index.append([i, j])
                edge_attr.append([dist])

    if len(edge_index) == 0:
        return None

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_attr, dtype=torch.float)
    y = torch.tensor([target_value], dtype=torch.float)

    return Data(z=z, edge_index=edge_index, edge_attr=edge_attr, y=y, num_nodes=len(z))

sample_graph = structure_to_graph(df_clean.iloc[0]["atoms"], df_clean.iloc[0][target], cutoff=5.0)
print(sample_graph)

## 11. Build Graph Dataset
To keep Colab training manageable, the first GNN experiment uses 5,000 cleaned structures. This can be increased later for stronger performance.

In [ ]:
MAX_SAMPLES = 5000
CUTOFF_RADIUS = 5.0

graph_dataset = []
for _, row in tqdm(df_clean.head(MAX_SAMPLES).iterrows(), total=MAX_SAMPLES):
    graph = structure_to_graph(row["atoms"], row[target], cutoff=CUTOFF_RADIUS)
    if graph is not None:
        graph_dataset.append(graph)

print("Total graphs created:", len(graph_dataset))

## 12. Train/Validation/Test Split and Target Normalization
The target scaler is fitted only on the training set. Validation data is used for early stopping, while the test set is kept for final evaluation.

In [ ]:
train_graphs, temp_graphs = train_test_split(graph_dataset, test_size=0.30, random_state=SEED)
val_graphs, test_graphs = train_test_split(temp_graphs, test_size=0.50, random_state=SEED)

gnn_target_scaler = StandardScaler()
train_targets = np.array([g.y.item() for g in train_graphs]).reshape(-1, 1)
gnn_target_scaler.fit(train_targets)

def normalize_graph_targets(graphs, scaler):
    normalized_graphs = []
    for graph in graphs:
        graph = graph.clone()
        y_norm = scaler.transform([[graph.y.item()]])[0][0]
        graph.y = torch.tensor([y_norm], dtype=torch.float)
        normalized_graphs.append(graph)
    return normalized_graphs

train_graphs_norm = normalize_graph_targets(train_graphs, gnn_target_scaler)
val_graphs_norm = normalize_graph_targets(val_graphs, gnn_target_scaler)
test_graphs_norm = normalize_graph_targets(test_graphs, gnn_target_scaler)

train_loader = DataLoader(train_graphs_norm, batch_size=32, shuffle=True)
val_loader = DataLoader(val_graphs_norm, batch_size=32, shuffle=False)
test_loader = DataLoader(test_graphs_norm, batch_size=32, shuffle=False)

print("Training graphs:", len(train_graphs_norm))
print("Validation graphs:", len(val_graphs_norm))
print("Testing graphs:", len(test_graphs_norm))

## 13. Improved Crystal GNN Architecture
The improved GNN uses learned atomic embeddings, Gaussian distance expansion, residual NNConv message passing, batch normalization, and mean/max graph pooling.

In [ ]:
class GaussianDistanceExpansion(nn.Module):
    # Expand scalar distances using Gaussian radial basis functions.
    def __init__(self, min_dist=0.0, max_dist=5.0, num_gaussians=50):
        super().__init__()
        centers = torch.linspace(min_dist, max_dist, num_gaussians)
        self.register_buffer("centers", centers)
        self.gamma = 10.0

    def forward(self, distances):
        distances = distances.view(-1, 1)
        return torch.exp(-self.gamma * (distances - self.centers) ** 2)


class ImprovedCrystalGNN(nn.Module):
    def __init__(self, max_atomic_number=100, embedding_dim=64, hidden_dim=128, edge_dim=50):
        super().__init__()
        self.atom_embedding = nn.Embedding(max_atomic_number + 1, embedding_dim)
        self.edge_expansion = GaussianDistanceExpansion(0.0, CUTOFF_RADIUS, edge_dim)
        self.node_encoder = nn.Linear(embedding_dim, hidden_dim)

        def edge_network():
            return nn.Sequential(
                nn.Linear(edge_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim * hidden_dim),
            )

        self.conv1 = NNConv(hidden_dim, hidden_dim, edge_network(), aggr="mean")
        self.conv2 = NNConv(hidden_dim, hidden_dim, edge_network(), aggr="mean")
        self.conv3 = NNConv(hidden_dim, hidden_dim, edge_network(), aggr="mean")

        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.bn3 = nn.BatchNorm1d(hidden_dim)

        self.regressor = nn.Sequential(
            nn.Linear(hidden_dim * 2, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 1),
        )

    def forward(self, data):
        z, edge_index, edge_attr, batch = data.z, data.edge_index, data.edge_attr, data.batch
        x = self.atom_embedding(z)
        x = F.relu(self.node_encoder(x))
        edge_feat = self.edge_expansion(edge_attr)

        h = F.relu(self.bn1(self.conv1(x, edge_index, edge_feat)))
        x = x + h
        h = F.relu(self.bn2(self.conv2(x, edge_index, edge_feat)))
        x = x + h
        h = F.relu(self.bn3(self.conv3(x, edge_index, edge_feat)))
        x = x + h

        pooled_mean = global_mean_pool(x, batch)
        pooled_max = global_max_pool(x, batch)
        graph_embedding = torch.cat([pooled_mean, pooled_max], dim=1)
        return self.regressor(graph_embedding).view(-1)

## 14. GNN Training Utilities
The validation set is used for early stopping. Final metrics are reported only on the held-out test set.

In [ ]:
def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0.0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        pred = model(batch)
        loss = loss_fn(pred, batch.y.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs
    return total_loss / len(loader.dataset)


def evaluate_gnn(model, loader, device, target_scaler):
    model.eval()
    y_true_norm, y_pred_norm = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            pred = model(batch)
            y_true_norm.extend(batch.y.view(-1).cpu().numpy())
            y_pred_norm.extend(pred.cpu().numpy())

    y_true_norm = np.array(y_true_norm).reshape(-1, 1)
    y_pred_norm = np.array(y_pred_norm).reshape(-1, 1)
    y_true = target_scaler.inverse_transform(y_true_norm).flatten()
    y_pred = target_scaler.inverse_transform(y_pred_norm).flatten()
    metrics = regression_report(y_true, y_pred)
    return metrics, y_true, y_pred

## 15. Train Improved Crystal GNN with Early Stopping
Early stopping prevents overfitting by saving the best validation model and stopping when validation MAE does not improve for a fixed number of epochs.

In [ ]:
model_gnn = ImprovedCrystalGNN(max_atomic_number=100, embedding_dim=64, hidden_dim=128, edge_dim=50).to(DEVICE)
optimizer = torch.optim.AdamW(model_gnn.parameters(), lr=5e-4, weight_decay=1e-4)
loss_fn = nn.MSELoss()

num_epochs = 100
patience = 15
best_val_mae = float("inf")
best_epoch = 0
counter = 0

history = {"epoch": [], "train_loss": [], "val_mae": [], "val_rmse": [], "val_r2": []}
os.makedirs("results/models", exist_ok=True)
best_model_path = "results/models/best_crystal_gnn.pth"

print("Training improved crystal GNN...")
for epoch in range(1, num_epochs + 1):
    train_loss = train_one_epoch(model_gnn, train_loader, optimizer, loss_fn, DEVICE)
    val_metrics, _, _ = evaluate_gnn(model_gnn, val_loader, DEVICE, gnn_target_scaler)

    val_mae = val_metrics["MAE"]
    val_rmse = val_metrics["RMSE"]
    val_r2 = val_metrics["R2"]

    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["val_mae"].append(val_mae)
    history["val_rmse"].append(val_rmse)
    history["val_r2"].append(val_r2)

    if val_mae < best_val_mae:
        best_val_mae = val_mae
        best_epoch = epoch
        counter = 0
        torch.save(model_gnn.state_dict(), best_model_path)
        status = "Best model saved"
    else:
        counter += 1
        status = f"Patience {counter}/{patience}"

    if epoch % 5 == 0 or status == "Best model saved":
        print(
            f"Epoch {epoch:03d} | Train Loss: {train_loss:.4f} | "
            f"Val MAE: {val_mae:.4f} | Val RMSE: {val_rmse:.4f} | "
            f"Val R²: {val_r2:.4f} | {status}"
        )

    if counter >= patience:
        print("Early stopping triggered.")
        break

print("Best epoch:", best_epoch)
print("Best validation MAE:", round(best_val_mae, 4))

## 16. Final Test Evaluation of Best GNN
The saved best model is reloaded and evaluated on the held-out test set.

In [ ]:
model_gnn.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
model_gnn.to(DEVICE)

gnn_test_metrics, y_true_gnn, y_pred_gnn = evaluate_gnn(model_gnn, test_loader, DEVICE, gnn_target_scaler)

print("===== Improved Crystal GNN Test Results =====")
print(f"Best Epoch : {best_epoch}")
print(f"MAE        : {gnn_test_metrics['MAE']:.4f} eV/atom")
print(f"RMSE       : {gnn_test_metrics['RMSE']:.4f}")
print(f"R²         : {gnn_test_metrics['R2']:.4f}")

## 17. Improved GNN Evaluation Plots
The parity plot checks overall prediction agreement, while the residual histogram shows the distribution of prediction errors.

In [ ]:
plot_parity(y_true_gnn, y_pred_gnn, "Improved Crystal GNN Parity Plot")
plot_residuals(y_true_gnn, y_pred_gnn, "Improved Crystal GNN Residual Distribution")

plt.figure(figsize=(8, 5))
plt.plot(history["epoch"], history["val_mae"])
plt.xlabel("Epoch")
plt.ylabel("Validation MAE")
plt.title("Improved GNN Validation MAE Curve")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 18. Model Comparison
The classical descriptor model and the crystal GNN are compared using MAE, RMSE, and R². Lower MAE/RMSE and higher R² indicate better performance.

In [ ]:
comparison_df = pd.DataFrame([
    {"Model": "XGBoost", "Input Representation": "Matminer composition descriptors", "MAE": xgb_mae, "RMSE": xgb_rmse, "R2": xgb_r2},
    {"Model": "Improved Crystal GNN", "Input Representation": "Crystal graph + atomic embeddings", "MAE": gnn_test_metrics["MAE"], "RMSE": gnn_test_metrics["RMSE"], "R2": gnn_test_metrics["R2"]},
])

display(comparison_df)

## 19. Save Models, Metrics, and Figures
This cell saves the trained models, scalers, metrics table, and final plots for GitHub packaging.

In [ ]:
os.makedirs("results/figures", exist_ok=True)
os.makedirs("results/tables", exist_ok=True)
os.makedirs("results/models", exist_ok=True)

joblib.dump(xgb_model, "results/models/xgboost_formation_energy.pkl")
joblib.dump(composition_scaler, "results/models/composition_feature_scaler.pkl")
joblib.dump(gnn_target_scaler, "results/models/gnn_target_scaler.pkl")

comparison_df.to_csv("results/tables/model_comparison.csv", index=False)
importance_df.to_csv("results/tables/xgboost_feature_importance.csv", index=False)

plt.figure(figsize=(7, 7))
plt.scatter(y_true_gnn, y_pred_gnn, alpha=0.5)
min_val = min(np.min(y_true_gnn), np.min(y_pred_gnn))
max_val = max(np.max(y_true_gnn), np.max(y_pred_gnn))
plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")
plt.xlabel("True Formation Energy")
plt.ylabel("Predicted Formation Energy")
plt.title("Improved Crystal GNN Parity Plot")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("results/figures/improved_gnn_parity_plot.png", dpi=300)
plt.show()

residuals_gnn = y_true_gnn - y_pred_gnn
plt.figure(figsize=(8, 5))
plt.hist(residuals_gnn, bins=60)
plt.xlabel("Residual Error")
plt.ylabel("Count")
plt.title("Improved Crystal GNN Residual Distribution")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("results/figures/improved_gnn_residual_distribution.png", dpi=300)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history["epoch"], history["val_mae"])
plt.xlabel("Epoch")
plt.ylabel("Validation MAE")
plt.title("Improved GNN Validation MAE Curve")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("results/figures/improved_gnn_validation_mae_curve.png", dpi=300)
plt.show()

print("Models, metrics, and figures saved successfully.")

## 20. Optional: Zip Outputs for Download
Run this cell in Colab to download the `results/` folder as a ZIP file.

In [ ]:
!zip -r MatPropNet_results.zip results

# Uncomment this in Google Colab to download:
# from google.colab import files
# files.download("MatPropNet_results.zip")

## Summary
This notebook developed a complete materials-property prediction framework. The XGBoost model provides a strong descriptor-based baseline, while the improved crystal GNN learns directly from atomic structure graphs. The workflow is ready to be modularized into a GitHub repository with separate scripts for preprocessing, feature engineering, model training, evaluation, and visualization.